In [2]:
import mysql.connector

def get_connection():
    return mysql.connector.connect(
        host='localhost',
        user='root',
        password='',
        database='db_kesehatan_mental'
    )

In [18]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.naive_bayes import MultinomialNB

conn = get_connection()

# load dataset
df_train = pd.read_sql("SELECT * FROM tb_data WHERE Jenis = 'Training'", conn)
df_test = pd.read_sql("SELECT * FROM tb_data WHERE Jenis = 'Testing'", conn)

for col in ['P1', 'P2', 'P3']:
    df_train[col] = df_train[col].astype(str).str.lower()
    df_test[col] = df_test[col].astype(str).str.lower()

# lakukan label encoding pada kelas
le_kelas = LabelEncoder()
df_train['Kelas'] = le_kelas.fit_transform(df_train['Kelas'])
df_test['Kelas'] = le_kelas.transform(df_test['Kelas'])

# buat data training dan testing
x_train = df_train[['P1', 'P2', 'P3']]
y_train = df_train['Kelas']
x_test = df_test[['P1', 'P2', 'P3']]
y_test = df_test['Kelas']

# lakukan label encoding untuk parameter
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
x_train = encoder.fit_transform(x_train)
x_test = encoder.transform(x_test)

# inisialisasi model
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(x_train, y_train)

# buat data testing
testing_raw = pd.DataFrame([['Normal', 'Sedang', 'Normal']], columns=['P1', 'P2', 'P3'])
for col in testing_raw.columns:
    testing_raw[col] = testing_raw[col].astype(str).str.lower()
testing_enc = encoder.transform(testing_raw)

# ambil prediksi
y_pred_nb = nb_model.predict(testing_enc)[0]

# maping kelas
kelas_map = {i: label for i, label in enumerate(le_kelas.classes_)}
prediksi = kelas_map[y_pred_nb]
prediksi

C:\Users\user\AppData\Local\Temp\ipykernel_1068\2127455599.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_train = pd.read_sql("SELECT * FROM tb_data WHERE Jenis = 'Training'", conn)
C:\Users\user\AppData\Local\Temp\ipykernel_1068\2127455599.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_test = pd.read_sql("SELECT * FROM tb_data WHERE Jenis = 'Testing'", conn)


'Normal'

In [25]:
from sklearn.model_selection import train_test_split

conn = get_connection()
cursor = conn.cursor()
# lakukan split
df = pd.read_sql("SELECT kd_data, P1, P2, P3, Kelas, Jenis FROM tb_data", conn)

# bagi parameter dan label
X = df[['P1', 'P2', 'P3']]
y = df['Kelas']

# split data menjadi 20%
X_train, X_test, _, _ = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ambil kd_data
train_ids = df.loc[X_train.index, 'kd_data'].tolist()
test_ids = df.loc[X_test.index, 'kd_data'].tolist()

# set db null
cursor.execute("UPDATE tb_data SET Jenis='null'")
conn.commit()

for kd in train_ids:
    cursor.execute("UPDATE tb_data SET Jenis='Training' WHERE kd_data=%s", (kd,))
for kd in test_ids:
    cursor.execute("UPDATE tb_data SET Jenis='Testing' WHERE kd_data=%s", (kd,))
conn.commit()
conn.close()

print({
    "message": "berhasil",
    "train_count": len(train_ids)
})

C:\Users\user\AppData\Local\Temp\ipykernel_1068\1167127832.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT kd_data, P1, P2, P3, Kelas, Jenis FROM tb_data", conn)


{'message': 'berhasil', 'train_count': 681}
